In [ ]:
%pip install timm tqdm torch torchvision opencv-python-headless ipywidgets jupyter

In [ ]:
import torch
import cv2
import numpy as np
import time
import os
from PIL import Image
from tqdm.auto import tqdm  # Use tqdm.auto instead of tqdm.notebook

# Available MiDaS models
AVAILABLE_MODELS = {
    'small': {'name': 'MiDaS_small', 'description': 'Fast, lightweight model (5MB)'},
    'medium': {'name': 'DPT_Hybrid', 'description': 'Medium quality-speed balance (470MB)'},
    'large': {'name': 'DPT_Large', 'description': 'Highest quality, but slower (1.4GB)'}
}

# Global model cache
loaded_models = {}
loaded_transforms = {}

# Initialize device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Using device: {device}")

def load_model(model_type='large'):
    """Load a specific MiDaS model"""
    if model_type not in AVAILABLE_MODELS:
        raise ValueError(f"Invalid model type: {model_type}. Available options: {', '.join(AVAILABLE_MODELS.keys())}")
    
    # Check if already loaded
    if model_type in loaded_models:
        return loaded_models[model_type], loaded_transforms[model_type]
    
    model_name = AVAILABLE_MODELS[model_type]['name']
    print(f"🔄 Loading MiDaS model: {model_name} ({AVAILABLE_MODELS[model_type]['description']})")
    
    try:
        # Load model
        print("Loading model...")
        model = torch.hub.load("intel-isl/MiDaS", model_name).to(device).eval()
        
        # Load transform
        print("Loading transforms...")
        midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
        
        # Select the appropriate transform
        if model_type == 'small':
            transform = midas_transforms.small_transform
        elif model_type == 'medium':
            transform = midas_transforms.dpt_hybrid_transform
        else:  # model_type == 'large'
            transform = midas_transforms.dpt_transform
        
        # Cache the models
        loaded_models[model_type] = model
        loaded_transforms[model_type] = transform
        print(f"✅ {model_name} loaded successfully!")
        
        return model, transform
    
    except Exception as e:
        print(f"❌ Error loading {model_name}: {str(e)}")
        raise e

# Initial default model load - small is fastest for most common use
try:
    default_model, default_transform = load_model('small')
except Exception as e:
    print(f"❌ Error loading default model: {str(e)}")

def estimate_depth(image_path: str, output_path: str = None, model_type='small', verbose=True):
    """Estimate depth using MiDaS
    
    Args:
        image_path: Path to input image
        output_path: Path to save depth map (optional)
        model_type: Which MiDaS model to use ('small', 'medium', 'large')
        verbose: Whether to show progress information
    
    Returns:
        Depth map as numpy array (uint8)
    """
    # Load the specified model if not using the default
    if model_type not in loaded_models:
        if verbose:
            print(f"Loading {model_type} model...")
        midas_model, transform = load_model(model_type)
    else:
        midas_model = loaded_models[model_type]
        transform = loaded_transforms[model_type]
    
    if verbose:
        print(f"🔄 Processing image with {model_type} model: {image_path}")
        
    start_time = time.time()
    
    if verbose:
        print("Loading image...")
    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img)  # Convert PIL Image to numpy array
    
    if verbose:
        print("Preprocessing...")
    input_batch = transform(img_np)
    input_batch = input_batch.to(device)
    
    if verbose:
        print("Running inference...")
    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img.size[::-1],  # (H, W)
            mode="bicubic",
            align_corners=False
        ).squeeze()
    
    if verbose:
        print("Processing depth map...")
    depth_map = prediction.cpu().numpy()
    
    # Normalize to 8-bit
    depth_min, depth_max = depth_map.min(), depth_map.max()
    depth_vis = (255 * (depth_map - depth_min) / (depth_max - depth_min)).astype(np.uint8)
    
    if output_path:
        if verbose:
            print("Saving result...")
        cv2.imwrite(output_path, depth_vis)
    
    # Calculate processing time
    elapsed_time = time.time() - start_time
    if verbose:
        print(f"✅ Depth estimation ({model_type}) completed in {elapsed_time:.2f} seconds")
    
    return depth_vis

if __name__ == "__main__":
    # Example usage
    depth = estimate_depth("your_image.jpg", "depth_output.png", model_type='small')
    cv2.imshow("Depth Map", depth)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

💻 Using device: cpu
🔄 Loading MiDaS model: MiDaS_small (Fast, lightweight model (5MB))
Loading model...


Using cache found in C:\Users\gm64x/.cache\torch\hub\intel-isl_MiDaS_master
c:\Users\gm64x\scoop\persist\miniconda3\envs\DepthRec\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
c:\Users\gm64x\scoop\persist\miniconda3\envs\DepthRec\Lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


Using cache found in C:\Users\gm64x/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master


Loading transforms...
✅ MiDaS_small loaded successfully!
🔄 Processing image with small model: your_image.jpg
Loading image...
Preprocessing...
Running inference...


Using cache found in C:\Users\gm64x/.cache\torch\hub\intel-isl_MiDaS_master


Processing depth map...
Saving result...
✅ Depth estimation (small) completed in 0.34 seconds


# Depth Estimation Using MiDaS

This notebook demonstrates how to use the MiDaS model for monocular depth estimation. MiDaS is a state-of-the-art model for predicting depth from a single RGB image.

MiDaS was trained on 10 distinct datasets with varying depth representations, allowing it to generalize well to unseen data.

## Available Model Options

### MiDaS Models (Intel)
- **Small** - Fast, lightweight model (5MB) - good for real-time or mobile applications
- **Medium** (DPT-Hybrid) - Balanced quality/speed (470MB) - good for most applications
- **Large** (DPT-Large) - Highest quality but slower (1.4GB) - best for offline processing

### Model Architecture Information
- **MiDaS** (by Intel) – Robust, supports multiple backends (PyTorch/ONNX)
- **DPT** (Dense Prediction Transformer) – More accurate but heavier, based on Vision Transformers
- The small model is based on a classic CNN architecture, while medium and large models use the more advanced DPT architecture

In [1]:
# Let's use the image from our workspace
import os
import matplotlib.pyplot as plt
import time
from datetime import datetime

def process_with_model_selection(image_path, model_type='small', show_result=True):
    """Process an image with a specific model type or all models"""
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    
    if model_type == 'all':
        results = {}
        fig = plt.figure(figsize=(18, 10)) if show_result else None
        
        # Track overall time
        all_start = time.time()
        
        # Process with each model type
        for i, model in enumerate(['small', 'medium', 'large']):
            print(f"\n{'='*50}\nProcessing with {model} model ({i+1}/3)\n{'='*50}")
            
            # Set output path
            output_path = os.path.join(os.path.dirname(image_path), f"depth_output_{model}.png")
            
            # Process image
            start = time.time()
            depth_map = estimate_depth(image_path, output_path, model_type=model)
            process_time = time.time() - start
            
            # Store result
            results[model] = {
                'depth_map': depth_map,
                'time': process_time,
                'output_path': output_path
            }
            
            if show_result:
                # Add to comparative plot
                ax = fig.add_subplot(2, 2, i+2)
                ax.imshow(depth_map, cmap='plasma')
                ax.set_title(f"{model.capitalize()} Model ({process_time:.2f}s)")
                ax.axis('off')
                plt.colorbar(ax.imshow(depth_map, cmap='plasma'), ax=ax, label='Depth')
        
        # Calculate total time
        total_time = time.time() - all_start
        print(f"\n✅ Processed with all models in {total_time:.2f} seconds total")
        
        # Add original image to plot
        if show_result:
            ax = fig.add_subplot(2, 2, 1)
            ax.imshow(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB))
            ax.set_title("Original Image")
            ax.axis('off')
            
            plt.tight_layout()
            plt.suptitle(f"MiDaS Depth Comparison - {base_name}", fontsize=16)
            plt.subplots_adjust(top=0.9)
            plt.show()
        
        return results
    else:
        # Process with a single model
        output_path = os.path.join(os.path.dirname(image_path), f"depth_output_{model_type}.png")
        
        # Process image
        depth_map = estimate_depth(image_path, output_path, model_type=model_type)
        
        # Show result if requested
        if show_result:
            plt.figure(figsize=(14, 6))
            
            plt.subplot(1, 2, 1)
            plt.title("Original Image")
            plt.imshow(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB))
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.title(f"Depth Map ({model_type} model)")
            plt.imshow(depth_map, cmap='plasma')
            plt.colorbar(label='Depth')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
        
        return depth_map

try:
    print(f"📊 Starting depth analysis at {datetime.now().strftime('%H:%M:%S')}")
    
    # Get the path to the sample image in the workspace
    image_path = os.path.join(os.getcwd(), "your_image.jpg")
    
    # Check if the image exists
    if os.path.exists(image_path):
        # Get image file size
        file_size = os.path.getsize(image_path) / (1024 * 1024)  # size in MB
        print(f"🖼️ Image found: {os.path.basename(image_path)} ({file_size:.2f} MB)")
        
        # Model selection section
        print("\n🔍 Available MiDaS models:")
        print("1. small - Fast, lightweight model (5MB)")
        print("2. medium - Medium quality-speed balance (470MB)")
        print("3. large - Highest quality, but slower (1.4GB)")
        print("4. all - Run all models for comparison")
        print("\nDefault is 'small' if no selection is made.")
        
        # For demonstration, let's use the small model
        # In a real notebook, you might want to use an input cell here
        model_choice = 'small'  # Options: 'small', 'medium', 'large', or 'all'
        print(f"\n🚀 Selected model: {model_choice}")
        
        # Process image with selected model(s)
        print("\n⏳ Starting depth estimation process...")
        start_time = time.time()
        result = process_with_model_selection(image_path, model_choice)
        process_time = time.time() - start_time
        
        print(f"⏱️ Total processing time: {process_time:.2f} seconds")
        
        # Display output information
        if model_choice == 'all':
            print("\n💾 Depth maps saved as:")
            for model in result:
                print(f"- {os.path.basename(result[model]['output_path'])} ({result[model]['time']:.2f}s)")
        else:
            output_path = os.path.join(os.path.dirname(image_path), f"depth_output_{model_choice}.png")
            print(f"\n💾 Depth map saved to: {output_path}")
        
        print(f"🏁 Process completed at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print(f"❌ Image not found at: {image_path}")
        print("Please upload an image named 'your_image.jpg' to the notebook directory")
        
        # Try to find any image files in the current directory
        image_files = [f for f in os.listdir(os.getcwd()) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        if image_files:
            print("\n📸 Available images in the current directory:")
            for img in image_files:
                img_size = os.path.getsize(os.path.join(os.getcwd(), img)) / (1024 * 1024)  # size in MB
                print(f"- {img} ({img_size:.2f} MB)")
            
            print("\n💡 You can use one of these images by running:")
            print(f"image_path = os.path.join(os.getcwd(), '[image-filename]')")
            print(f"result = process_with_model_selection(image_path, 'small')  # or 'medium', 'large', 'all'")
except Exception as e:
    print(f"❌ An error occurred: {str(e)}")
    print("\n💻 Please make sure you have installed all the required packages:")
    print("%pip install torch torchvision timm opencv-python pillow matplotlib tqdm")

📊 Starting depth analysis at 13:17:37
🖼️ Image found: your_image.jpg (0.07 MB)

🔍 Available MiDaS models:
1. small - Fast, lightweight model (5MB)
2. medium - Medium quality-speed balance (470MB)
3. large - Highest quality, but slower (1.4GB)
4. all - Run all models for comparison

Default is 'small' if no selection is made.

🚀 Selected model: small

⏳ Starting depth estimation process...
❌ An error occurred: name 'estimate_depth' is not defined

💻 Please make sure you have installed all the required packages:
%pip install torch torchvision timm opencv-python pillow matplotlib tqdm


In [2]:
# Interactive model selection
from ipywidgets import interact, widgets

def select_model_and_image(model='small', image_file=None):
    """Process an image with the selected model
    
    Args:
        model: Model type to use ('small', 'medium', 'large', or 'all')
        image_file: Image file to process (from dropdown)
    """
    if image_file is None:
        print("❌ Please select an image to process")
        return None
    
    # Find available images
    try:
        image_path = os.path.join(os.getcwd(), image_file)
        if not os.path.exists(image_path):
            print(f"❌ Image file not found: {image_file}")
            return None
            
        # Process and display results
        print(f"🔍 Processing {os.path.basename(image_path)} with {model} model...")
        return process_with_model_selection(image_path, model)
        
    except Exception as e:
        print(f"❌ An error occurred: {str(e)}")
        return None

# Find available images
image_files = [f for f in os.listdir(os.getcwd()) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]

# Create the interactive widget
if image_files:
    # Set default image to your_image.jpg if available
    default_image = "your_image.jpg" if "your_image.jpg" in image_files else image_files[0]
    
    interact(
        select_model_and_image,
        model=widgets.Dropdown(
            options=[('Small (fast)', 'small'), 
                     ('Medium (balanced)', 'medium'), 
                     ('Large (best quality)', 'large'),
                     ('All Models (comparison)', 'all')],
            value='small',
            description='MiDaS Model:'
        ),
        image_file=widgets.Dropdown(
            options=image_files,
            value=default_image,
            description='Input Image:'
        )
    )
else:
    print("❌ No image files found in the current directory. Please upload at least one image.")

interactive(children=(Dropdown(description='MiDaS Model:', options=(('Small (fast)', 'small'), ('Medium (balan…

In [3]:
# Function to browse available images
def browse_images():
    """Browse and display available images in the current directory"""
    import matplotlib.pyplot as plt
    from ipywidgets import interact, widgets
    
    # Find available images
    image_files = [f for f in os.listdir(os.getcwd()) 
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
    
    if not image_files:
        print("❌ No image files found in the current directory.")
        return None
    
    # Add a 'None' option to not display any image
    display_options = [('None', None)] + [(f, f) for f in image_files]
    
    # Function to display the selected image
    def show_image(filename):
        if filename is None:
            return
            
        image_path = os.path.join(os.getcwd(), filename)
        img = plt.imread(image_path)
        
        # Get file size in MB
        file_size = os.path.getsize(image_path) / (1024 * 1024)
        
        # Print image info
        print(f"📊 Image: {filename}")
        print(f"🖼️ Size: {img.shape[1]}×{img.shape[0]} pixels")
        print(f"📦 File size: {file_size:.2f} MB")
        
        # Display the image
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Image Preview: {filename}")
        plt.show()
        
        # Return image path for convenience
        return image_path
    
    # Create the interactive widget
    return interact(
        show_image,
        filename=widgets.Dropdown(
            options=display_options,
            value=None,
            description='Select image:'
        )
    )

# Run the function to browse images
browse_images()

interactive(children=(Dropdown(description='Select image:', options=(('None', None), ('depth_output.png', 'dep…

<function __main__.browse_images.<locals>.show_image(filename)>

## MiDaS and DPT Model Architectures

### MiDaS Models
MiDaS (Mixing Datasets for depth estimation) is a family of models developed by Intel for monocular depth estimation. These models have been trained on 10+ diverse datasets to achieve robust performance across various scenarios.

#### Model Options Explained

1. **Small Model (MiDaS_small):**
   - **Architecture:** Efficient CNN with MobileNet-like design
   - **Size:** ~5MB
   - **Performance:** Optimized for speed (real-time inference)
   - **Use Cases:** Mobile applications, drones, or any scenario requiring real-time processing
   - **Framework Support:** PyTorch, ONNX export, OpenVINO
   - **Trade-offs:** Lower accuracy, especially with complex scenes or fine details

2. **Medium Model (DPT_Hybrid):**
   - **Architecture:** Dense Prediction Transformer (DPT) with hybrid CNN-ViT backbone
   - **Size:** ~470MB
   - **Performance:** Balanced speed/quality compromise
   - **Use Cases:** General applications needing good depth accuracy without extreme processing requirements
   - **Technical Details:** Uses a Vision Transformer backbone with efficient distillation tokens

3. **Large Model (DPT_Large):**
   - **Architecture:** Dense Prediction Transformer (DPT) with full ViT-Large backbone
   - **Size:** ~1.4GB
   - **Performance:** Highest quality depth maps with fine details
   - **Use Cases:** Professional applications, offline processing, research
   - **Technical Details:** Leverages the ViT-Large architecture with additional transformer blocks

### Technical Background

**Dense Prediction Transformer (DPT)** architecture connects transformers with a convolutional neural network decoder to produce high-resolution depth predictions. Unlike CNNs, which struggle with global context due to limited receptive fields, transformers excel at capturing long-range dependencies through their self-attention mechanism.

**Key Innovations:**

- **Reassembly blocks:** Combine local and global features effectively
- **Multi-head attention:** Enables parallel processing of spatial relationships
- **Skip connections:** Preserve spatial details for higher resolution outputs

For more information on the technical details, see the [MiDaS GitHub repository](https://github.com/isl-org/MiDaS) and the [DPT paper](https://arxiv.org/abs/2103.13413).

In [ ]:
# Quick model selection - run this cell and input your choice
model_choice = input('Choose model (small/medium/large/all): ').lower()
if model_choice not in ['small', 'medium', 'large', 'all']:
    print("Invalid choice. Using 'small' as default.")
    model_choice = 'small'

# Process the image with the selected model
image_path = os.path.join(os.getcwd(), "your_image.jpg")
if os.path.exists(image_path):
    result = process_with_model_selection(image_path, model_choice)
else:
    print("Please place an image named 'your_image.jpg' in the notebook directory")

NameError: name 'estimate_depth' is not defined

In [7]:
# Utility function to compare model outputs
def compare_depth_outputs(image_path=None):
    """Compare depth outputs from different MiDaS models
    
    Args:
        image_path: Path to input image (optional, will prompt if None)
    """
    import matplotlib.pyplot as plt
    from matplotlib import cm
    
    # Find available output images
    output_files = {}
    for model_type in ['small', 'medium', 'large']:
        output_name = f"depth_output_{model_type}.png"
        output_path = os.path.join(os.getcwd(), output_name)
        if os.path.exists(output_path):
            output_files[model_type] = output_path
    
    if not output_files:
        print("❌ No depth output files found. Please run models first.")
        return
        
    # Select original image
    if image_path is None:
        # Find available images
        image_files = [f for f in os.listdir(os.getcwd()) 
                    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        
        if not image_files:
            print("❌ No image files found in the current directory.")
            return
            
        # Use the first image if your_image.jpg isn't available
        if "your_image.jpg" in image_files:
            image_path = os.path.join(os.getcwd(), "your_image.jpg")
        else:
            image_path = os.path.join(os.getcwd(), image_files[0])
    
    # Create comparison plot
    fig = plt.figure(figsize=(15, 10))
    
    # Original image
    ax = fig.add_subplot(2, 2, 1)
    original_img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    ax.imshow(original_img)
    ax.set_title("Original Image")
    ax.axis('off')
    
    # Depth outputs
    cmap = cm.get_cmap('plasma')  # Consistent colormap across all images
    
    for i, (model_type, output_path) in enumerate(output_files.items(), start=2):
        depth_map = cv2.imread(output_path, cv2.IMREAD_GRAYSCALE)
        
        if depth_map is not None:
            ax = fig.add_subplot(2, 2, i)
            im = ax.imshow(depth_map, cmap='plasma')
            ax.set_title(f"{model_type.capitalize()} Model")
            ax.axis('off')
            plt.colorbar(im, ax=ax, label='Depth')
    
    plt.tight_layout()
    plt.suptitle("MiDaS Depth Comparison", fontsize=16, y=0.98)
    plt.subplots_adjust(top=0.90)
    plt.show()

# Run the comparison function if at least one output exists
try:
    at_least_one = any(os.path.exists(os.path.join(os.getcwd(), f"depth_output_{model}.png")) 
                       for model in ['small', 'medium', 'large'])
    if at_least_one:
        compare_depth_outputs()
    else:
        print("❌ No depth output files found yet. Process at least one image first.")
except Exception as e:
    print(f"❌ Error comparing outputs: {str(e)}")


❌ No depth output files found yet. Process at least one image first.


In [8]:
# Batch processing with progress tracking

def process_batch_with_eta(image_paths, output_dir=None):
    """Process multiple images with progress tracking and ETA"""
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    results = {}
    start_time = time.time()
    
    print(f"🔄 Starting batch processing of {len(image_paths)} images at {datetime.now().strftime('%H:%M:%S')}")
    
    # Process each image with progress tracking
    for i, img_path in enumerate(tqdm(image_paths, desc="Batch progress")):
        img_name = os.path.basename(img_path)
        
        # Calculate ETA
        if i > 0:
            elapsed = time.time() - start_time
            images_per_second = i / elapsed
            remaining_images = len(image_paths) - i
            eta_seconds = remaining_images / images_per_second
            eta_min = int(eta_seconds // 60)
            eta_sec = int(eta_seconds % 60)
            print(f"Processing {i+1}/{len(image_paths)}: {img_name} (ETA: {eta_min}m {eta_sec}s)")
        else:
            print(f"Processing {i+1}/{len(image_paths)}: {img_name}")
        
        # Set output path if provided
        output_path = None
        if output_dir:
            output_name = f"{os.path.splitext(img_name)[0]}_depth.png"
            output_path = os.path.join(output_dir, output_name)
        
        # Process image with minimal verbosity (progress bars are handled by the batch tracker)
        depth = estimate_depth(img_path, output_path, verbose=False)
        results[img_path] = depth
    
    # Calculate statistics
    total_time = time.time() - start_time
    avg_time = total_time / len(image_paths)
    
    print(f"✅ Batch processing completed in {total_time:.2f} seconds")
    print(f"📊 Average processing time per image: {avg_time:.2f} seconds")
    print(f"🏁 Finished at {datetime.now().strftime('%H:%M:%S')}")
    
    return results

# Example usage (uncomment to run):
"""
# Find all images in the current directory
all_images = [os.path.join(os.getcwd(), f) for f in os.listdir(os.getcwd()) 
             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

# Create output directory
output_dir = os.path.join(os.getcwd(), "depth_results")

# Run batch processing
results = process_batch_with_eta(all_images, output_dir)
print(f"Processed {len(results)} images. Results saved to {output_dir}")
"""

'\n# Find all images in the current directory\nall_images = [os.path.join(os.getcwd(), f) for f in os.listdir(os.getcwd()) \n             if f.lower().endswith((\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\'))]\n\n# Create output directory\noutput_dir = os.path.join(os.getcwd(), "depth_results")\n\n# Run batch processing\nresults = process_batch_with_eta(all_images, output_dir)\nprint(f"Processed {len(results)} images. Results saved to {output_dir}")\n'

## Advanced Model Options

MiDaS offers several model variants optimized for different use cases:

### Model Types:

1. **MiDaS** (Original Intel Models)
   - Versatile models that work well across various scenes
   - Can be exported to ONNX for deployment on different hardware

2. **DPT** (Dense Prediction Transformers)
   - Based on Vision Transformer architecture
   - Superior quality but more computationally intensive
   - Medium and Large models in this notebook use the DPT architecture

3. **Lightweight Models** (not included in this notebook)
   - FastDepth - Optimized for embedded systems and mobile devices
   - MobileNetV2 backends - For real-time applications

### Export and Deployment
To deploy these models in production environments, consider:
- Converting to ONNX for cross-platform compatibility
- Using TensorRT for NVIDIA GPU acceleration
- Using CoreML for Apple devices
- Quantization for reduced model size

In [ ]:
# Browse and select images from the current directory

def browse_images():
    """Display all images in the current directory with thumbnails"""
    image_files = [f for f in os.listdir(os.getcwd()) 
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
    
    if not image_files:
        print("❌ No image files found in the current directory.")
        return
    
    print(f"📸 Found {len(image_files)} images in the current directory:")
    
    # Create a grid of thumbnails
    num_images = len(image_files)
    cols = min(4, num_images)
    rows = (num_images + cols - 1) // cols
    
    fig = plt.figure(figsize=(16, 4 * rows))
    
    for i, img_name in enumerate(image_files):
        img_path = os.path.join(os.getcwd(), img_name)
        img_size = os.path.getsize(img_path) / (1024 * 1024)  # size in MB
        
        # Read and display image
        ax = fig.add_subplot(rows, cols, i+1)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{img_name}\n({img_size:.2f} MB)")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Return list of images
    return image_files

# Execute to see available images
# browse_images()

In [ ]:
# Process video frames with live progress updates

def process_video_frames(video_path, output_dir=None, sample_rate=10):
    """Process frames from a video with progress visualization
    
    Args:
        video_path: Path to the video file
        output_dir: Directory to save depth maps
        sample_rate: Process every Nth frame (higher values = faster processing, fewer frames)
    """
    if not os.path.exists(video_path):
        print(f"❌ Video not found: {video_path}")
        return
    
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Error opening video file")
        return
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    processed_frames = total_frames // sample_rate
    
    print(f"🎥 Video info: {total_frames} frames, {fps:.2f} FPS, Duration: {duration:.2f}s")
    print(f"⚙️ Processing every {sample_rate}th frame ({processed_frames} frames total)")
    
    frame_count = 0
    processed_count = 0
    results = []
    
    start_time = time.time()
    pbar = tqdm(total=processed_frames, desc="Video processing")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Process every Nth frame
        if frame_count % sample_rate == 0:
            # Convert frame for processing
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            
            # Save frame to temporary file (MiDaS function expects a file path)
            temp_frame_path = os.path.join(os.getcwd(), "_temp_frame.jpg")
            img.save(temp_frame_path)
            
            # Process frame
            if output_dir:
                output_path = os.path.join(output_dir, f"frame_{processed_count:04d}_depth.png")
            else:
                output_path = None
                
            # Process with minimal verbosity
            depth_map = estimate_depth(temp_frame_path, output_path, verbose=False)
            results.append(depth_map)
            
            # Clean up temp file
            if os.path.exists(temp_frame_path):
                os.remove(temp_frame_path)
                
            # Update progress
            processed_count += 1
            current_time = time.time() - start_time
            frames_per_second = processed_count / current_time
            eta_seconds = (processed_frames - processed_count) / frames_per_second
            
            pbar.set_postfix({"FPS": f"{frames_per_second:.2f}", "ETA": f"{eta_seconds/60:.2f}m"})
            pbar.update(1)
            
        frame_count += 1
    
    # Close progress bar and video
    pbar.close()
    cap.release()
    
    # Calculate statistics
    total_time = time.time() - start_time
    fps_processing = processed_count / total_time
    
    print(f"✅ Video processing completed in {total_time:.2f} seconds")
    print(f"📊 Average processing speed: {fps_processing:.2f} frames per second")
    print(f"🏁 Finished at {datetime.now().strftime('%H:%M:%S')}")
    
    return results

# Example usage (uncomment to run):
"""
# Process a video file
video_path = "your_video.mp4"  # Change to your video file
if os.path.exists(video_path):
    output_dir = os.path.join(os.getcwd(), "video_depth_maps")
    depth_frames = process_video_frames(video_path, output_dir, sample_rate=30)
    print(f"Processed {len(depth_frames)} frames. Results saved to {output_dir}")
else:
    print(f"Video file not found: {video_path}")
"""